In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.advisor import PRICE_INPUTS, advise, fit_price_model
from src.day_profile import build_typical_days, classify_days, make_day
from src.load import load_profile
from src.validation import split_train_test

master = pd.read_csv(ROOT / "data" / "processed" / "master_hourly.csv", index_col=0, parse_dates=True)
features = pd.read_csv(ROOT / "data" / "processed" / "features_h72.csv", index_col=0, parse_dates=True)
profile = load_profile(ROOT / "data" / "raw" / "rbee extract by hour and state.csv")

library = build_typical_days(master)
price_model = fit_price_model(features)

day = make_day(library, month=7, sky="Overcast", tmin=4, tmax=10, wind_10m=6.0)
result = advise(day, price_model, profile)
result["hourly"].round(2)

,hour,price_mwh,solar_kwh,load_kwh,battery_pct_B,battery_pct_C,action,battery_kw,import_kwh,export_kwh
hour_end,,,,,,,,,,
2025-07-02 01:00:00,0,28.04,0.00,0.52,44.44,49.95,Use grid power,0.00,0.52,0.0
2025-07-02 02:00:00,1,19.10,0.00,0.47,39.48,49.90,Use grid power,0.00,0.47,0.0
2025-07-02 03:00:00,2,12.59,0.00,0.43,34.90,52.62,Charge battery from the grid,0.29,0.72,0.0
2025-07-02 04:00:00,3,11.59,0.00,0.41,30.50,100.00,Charge battery from the grid,5.00,5.41,0.0
2025-07-02 05:00:00,4,15.61,0.00,0.42,26.08,100.00,Use grid power,0.01,0.43,0.0
2025-07-02 06:00:00,5,23.74,0.00,0.49,20.91,100.00,Use grid power,0.01,0.50,0.0
2025-07-02 07:00:00,6,36.64,0.00,0.69,13.58,99.90,Use grid power,0.00,0.69,0.0
2025-07-02 08:00:00,7,56.35,0.04,0.88,4.64,91.72,Discharge battery to the home,-0.77,0.08,0.0
2025-07-02 09:00:00,8,57.28,0.27,0.82,0.00,85.80,Discharge battery to the home,-0.55,0.00,0.0


In [2]:
cols = ["strategy", "import_cost", "export_income", "supply_charge", "total", "battery_end_pct"]
print("A day like this, when it repeats:")
display(result["costs_repeating"][cols].round(2))
print("Today, starting from 50% charge:")
display(result["costs_today"][cols].round(2))

A day like this, when it repeats:


,strategy,import_cost,export_income,supply_charge,total,battery_end_pct
A,"No battery (solar, then grid)",3.54,0.04,1.21,4.71,NaN
B,"Simple rule (store surplus solar, never charge...",3.39,0.00,1.21,4.60,0.0
C,Forecast-driven plan (recommended),2.56,0.00,1.21,3.77,0.0


Today, starting from 50% charge:


,strategy,import_cost,export_income,supply_charge,total,battery_end_pct
A,"No battery (solar, then grid)",3.54,0.04,1.21,4.71,NaN
B,"Simple rule (store surplus solar, never charge...",2.44,0.00,1.21,3.65,0.0
C,Forecast-driven plan (recommended),1.61,0.00,1.21,2.83,0.0


In [3]:
from src.controllers import controller_plan
from src.dispatch import annual_bill
from src.metrics import mae

TEST_START, HORIZON = pd.Timestamp("2025-10-01"), 72
train, test = split_train_test(features, TEST_START, HORIZON)
model = fit_price_model(train)                         # trained only on data before the test year
full = pd.Series(model.predict(test[PRICE_INPUTS]), index=test.index)

library_before = build_typical_days(master[master.index <= TEST_START - pd.Timedelta(hours=HORIZON)])
days, _ = classify_days(master)
simplified = pd.Series(np.nan, index=test.index)
for day_start, row in days.iterrows():
    if day_start < TEST_START - pd.Timedelta(days=1):
        continue
    try:
        d = make_day(library_before, int(row["month"]), row["sky"], row["tmin"], row["tmax"],
                     row["wind"], weekend=day_start.dayofweek >= 5)
    except ValueError:
        continue
    hour_end = day_start + pd.to_timedelta(d["hour"].to_numpy() + 1, unit="h")
    forecast = pd.Series(model.predict(d[PRICE_INPUTS]), index=hour_end)
    shared = simplified.index.intersection(forecast.index)
    simplified.loc[shared] = forecast.loc[shared]
print("hours rebuilt from a simple description:", simplified.notna().sum(), "of", len(simplified))
simplified = simplified.fillna(full)

fc = pd.read_csv(ROOT / "results" / "forecasts.csv", index_col=0, parse_dates=True)
pv = pd.read_csv(ROOT / "data" / "processed" / "pv_hourly.csv", index_col=0, parse_dates=True)["pv_kwh"]
load = pd.read_csv(ROOT / "data" / "processed" / "household_load_hourly.csv",
                   index_col=0, parse_dates=True)["load_kwh"]
spot, n_days = fc["price_actual_mwh"].values, len(fc) / 24

def bill(prices):
    run = controller_plan(pv, load, prices.reindex(fc.index).values)
    return annual_bill(run["import_kwh"], run["export_kwh"], spot, n_days)["total"]

summary = pd.DataFrame({
    "Price MAE ($/MWh)": [mae(test["price_mwh"], full), mae(test["price_mwh"], simplified)],
    "Controller C annual bill ($)": [bill(full), bill(simplified)]},
    index=["Full hourly weather", "Simplified day description"])
summary.round(2).to_csv(ROOT / "results" / "day_profile_validation.csv")
summary.round(2)

hours rebuilt from a simple description: 8497 of 8616


,Price MAE ($/MWh),Controller C annual bill ($)
Full hourly weather,37.77,601.95
Simplified day description,41.73,608.54
